# 01 — DG38 Classification Baselines & 3-Seed Experiments (ResNet50 / DINOv2 ViT-B/14 / ViT-B/16)

**Thứ tự ưu tiên theo kế hoạch DG38:**
1. **Ưu tiên số 1 — ResNet50 Baseline (`resnet50`)**:
   - Huấn luyện trên đúng tập Classification cố định: **9,807 Train / 1,226 Val** (10 lớp, trong `/content/dg38_data/classification/standardized_256`), chọn checkpoint theo **Validation Macro F1**.
   - Chạy Pilot 20 batches ở Cell 2 để đo tốc độ thực tế trên GPU hiện tại và dự báo CU trước khi chạy đầy đủ 15 epochs ở Cell 3.
   - Sau khi hoàn thành Seed `42`, tiếp tục chạy Seed `123` và `2026` để có **Mean ± Sample Std (`ddof=1`)**.
2. **Mở rộng 3 Seeds cho DINOv2 ViT-B/14 (`dinov2_vitb14`, fine-tune 4 block cuối)**:
   - Chạy `SEED = 123` và `SEED = 2026` khi ngân sách trong nhóm `three_seed_experiments` còn đủ.
3. **Đánh giá trên Tập Test Đã Khoá (`1,226` ảnh, Cell 4)**:
   - Tập Test (`dg38_cls_test_locked.zip`) chỉ được giải nén và đánh giá sau khi quá trình huấn luyện + chọn best checkpoint trên Validation đã hoàn tất (`RUN_LOCKED_TEST_EVAL = True`).

> **An toàn CU:** Biến `RUN_FULL_TRAINING` mặc định là `False` và `OBSERVED_CU_PER_HOUR` bắt buộc do người dùng nhập từ giao diện Colab (`Runtime -> View resources`).

In [ ]:
# [CELL 1] Thiết lập môi trường & tham số thí nghiệm Classification
import os
import sys
import zipfile
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

DRIVE_ROOT = Path(os.environ.get("DG38_DRIVE_ROOT", "/content/drive/MyDrive/DG38_Colab_Package"))
REPO_ROOT = DRIVE_ROOT / "code" if (DRIVE_ROOT / "code" / "model.py").exists() else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXP_DIR = REPO_ROOT / "evaluation_runs" / "dg38_additional_experiments"
MANIFEST_PATH = EXP_DIR / "manifests" / "classification_train_val_manifest.json"
TEST_MANIFEST_PATH = EXP_DIR / "manifests" / "classification_test_manifest.json"
DATA_ROOT = Path(os.environ.get("DG38_CLS_DATA_ROOT", "/content/dg38_data/classification/standardized_256"))
ARTIFACTS_DIR = (DRIVE_ROOT / "artifacts") if DRIVE_ROOT.exists() else EXP_DIR

# ================= THAM SỐ THÍ NGHIỆM =================
ARCH = "resnet50"                # Lựa chọn: "resnet50" | "dinov2_vitb14" | "vit_b_16"
SEED = 42                        # Lựa chọn: 42 | 123 | 2026
UNFREEZE_BLOCKS = 4              # Chỉ áp dụng cho dinov2_vitb14 (mặc định 4 block cuối)
EPOCHS = 15                      # 15 epochs chuẩn
LR = 3e-4 if ARCH == "resnet50" else 1e-4

# ================= CỔNG KIỂM SOÁT NGÂN SÁCH CU =================
OBSERVED_CU_PER_HOUR = None      # <-- BẮT BUỘC: Nhập Usage Rate thực tế từ Colab UI (ví dụ: 1.48)
BUDGET_CATEGORY = "resnet50_baseline" if (ARCH == "resnet50" and SEED == 42) else "three_seed_experiments"
RUN_FULL_TRAINING = False        # ĐỔI THÀNH True SAU KHI XEM BẢNG DỰ BÁO CU Ở CELL 2 ĐẠT APPROVED
RUN_LOCKED_TEST_EVAL = False     # ĐỔI THÀNH True Ở CELL 4 SAU KHI ĐÃ TRAIN XONG VÀ CHỌN BEST CHECKPOINT TRÊN VAL

if OBSERVED_CU_PER_HOUR is None or float(OBSERVED_CU_PER_HOUR) <= 0:
    raise ValueError("Vui lòng nhập OBSERVED_CU_PER_HOUR (Usage Rate từ 'Runtime -> View resources') trước khi chạy!")

print(f"DATA_ROOT     : {DATA_ROOT} (exists={DATA_ROOT.exists()})")
print(f"MANIFEST_PATH : {MANIFEST_PATH} (exists={MANIFEST_PATH.exists()})")

In [ ]:
# [CELL 2] Chạy Pilot 20 Batches để đo tốc độ thực tế & Kiểm tra Cổng Ngân Sách (CU Gate)
trainer_script = EXP_DIR / "dg38_cls_trainer.py"
out_dir = ARTIFACTS_DIR / "runs" / f"cls_{ARCH}_blocks{UNFREEZE_BLOCKS if ARCH=='dinov2_vitb14' else 'all'}_seed{SEED}"

cmd_pilot = [
    sys.executable,
    str(trainer_script),
    "--mode", "pilot",
    "--arch", ARCH,
    "--unfreeze-blocks", str(UNFREEZE_BLOCKS),
    "--seed", str(SEED),
    "--epochs", str(EPOCHS),
    "--lr", str(LR),
    "--data-root", str(DATA_ROOT),
    "--manifest", str(MANIFEST_PATH),
    "--output-dir", str(out_dir),
    "--cu-per-hour", str(OBSERVED_CU_PER_HOUR),
    "--budget-category", BUDGET_CATEGORY,
]
print("Đang chạy Pilot Benchmark:", " ".join(cmd_pilot))
if DATA_ROOT.exists():
    res = subprocess.run(cmd_pilot, text=True)
    if res.returncode != 0:
        raise RuntimeError("Pilot Benchmark gặp lỗi.")
else:
    print(f"[INFO] Không tìm thấy {DATA_ROOT}. Hãy chạy Cell 4 của Notebook 00 để giải nén dg38_cls_train_val.zip.")

In [ ]:
# [CELL 3] Huấn luyện đầy đủ trên Train/Val (Hỗ trợ Resume tự động nếu phiên Colab bị ngắt) & Ghi nhận CU thực tế
if not RUN_FULL_TRAINING:
    print("[DỪNG AN TOÀN] RUN_FULL_TRAINING đang là False. Sau khi kiểm tra dự báo CU ở Cell 2 đạt approved=True, hãy đặt RUN_FULL_TRAINING = True rồi chạy lại Cell này.")
else:
    cmd_train = [
        sys.executable,
        str(trainer_script),
        "--mode", "train",
        "--arch", ARCH,
        "--unfreeze-blocks", str(UNFREEZE_BLOCKS),
        "--seed", str(SEED),
        "--epochs", str(EPOCHS),
        "--lr", str(LR),
        "--data-root", str(DATA_ROOT),
        "--manifest", str(MANIFEST_PATH),
        "--output-dir", str(out_dir),
        "--cu-per-hour", str(OBSERVED_CU_PER_HOUR),
        "--budget-category", BUDGET_CATEGORY,
        "--resume",
    ]
    res = subprocess.run(cmd_train, text=True)
    if res.returncode != 0:
        raise RuntimeError("Quá trình huấn luyện gặp lỗi hoặc bị chặn bởi cổng ngân sách CU.")

In [ ]:
# [CELL 4] Đánh giá Checkpoint tốt nhất (đã chọn theo Validation Macro F1) trên Tập Test Đã Khoá (1,226 ảnh)
if not RUN_LOCKED_TEST_EVAL:
    print("[DỪNG AN TOÀN] RUN_LOCKED_TEST_EVAL đang là False. Chỉ bật True sau khi Cell 3 đã huấn luyện xong và khoá best_model.pt.")
else:
    best_ckpt = out_dir / "best_model.pt"
    if not best_ckpt.exists():
        raise FileNotFoundError(f"Chưa có best_model.pt tại {best_ckpt}. Hãy hoàn thành huấn luyện ở Cell 3 trước.")

    # Giải nén gói Test đã khoá nếu chưa giải nén
    test_zip = DRIVE_ROOT / "zips" / "dg38_cls_test_locked.zip"
    if test_zip.exists():
        print(f"Đang giải nén gói Test {test_zip} -> {DATA_ROOT.parent} ...")
        with zipfile.ZipFile(test_zip, "r") as zf:
            zf.extractall(DATA_ROOT.parent)

    cmd_eval_test = [
        sys.executable,
        str(trainer_script),
        "--mode", "eval-test",
        "--arch", ARCH,
        "--unfreeze-blocks", str(UNFREEZE_BLOCKS),
        "--seed", str(SEED),
        "--data-root", str(DATA_ROOT),
        "--manifest", str(MANIFEST_PATH),
        "--test-manifest", str(TEST_MANIFEST_PATH),
        "--checkpoint", str(best_ckpt),
        "--output-dir", str(out_dir),
    ]
    res = subprocess.run(cmd_eval_test, text=True)
    if res.returncode != 0:
        raise RuntimeError("Đánh giá trên tập Test gặp lỗi.")